# 06 Model Selection (DuckDB Input + BLOB Artifacts)

Compare candidate models using DuckDB input data and store the best model as a DuckDB BLOB artifact.

In [ ]:
from pathlib import Path
import json
import pickle

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

from utils.duckdb_utils import init_ml_artifacts_table, open_duckdb, query_df
from utils.paths import get_db_paths, resolve_workspace
            


In [ ]:
workspace = resolve_workspace(Path.cwd())
input_db, output_db = get_db_paths(workspace)

init_ml_artifacts_table(output_db)
            


In [ ]:
raw = query_df(
    input_db,
    '''
    SELECT file, COALESCE(NULLIF(text, ''), description) AS text, status, day, level_name, conference
    FROM sessions_in_preprocessed
    WHERE COALESCE(NULLIF(text, ''), description) IS NOT NULL
    '''
)

target_col = next((c for c in ['status', 'day', 'level_name', 'conference'] if raw[c].fillna('Unknown').nunique() > 1), None)
if target_col is None:
    raise ValueError('No usable target column found.')

df = raw[['text', target_col]].copy()
df[target_col] = df[target_col].fillna('Unknown')
counts = df[target_col].value_counts()
df = df[df[target_col].isin(counts[counts >= 2].index)].copy()

X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df[target_col], test_size=0.25, random_state=42, stratify=df[target_col]
)

models = {
    'logreg': LogisticRegression(max_iter=2000, class_weight='balanced'),
    'linear_svc': LinearSVC(),
    'rf': RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for name, model in models.items():
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)),
        ('model', model),
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1_weighted', n_jobs=-1)
    rows.append({'model': name, 'cv_f1_mean': float(scores.mean()), 'cv_f1_std': float(scores.std())})

results_df = pd.DataFrame(rows).sort_values('cv_f1_mean', ascending=False).reset_index(drop=True)
best_model_name = results_df.loc[0, 'model']

best_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)),
    ('model', models[best_model_name]),
])
best_pipe.fit(X_train, y_train)
pred = best_pipe.predict(X_test)
holdout_f1 = f1_score(y_test, pred, average='weighted')

metrics = {
    'target_col': target_col,
    'best_model': best_model_name,
    'holdout_f1_weighted': float(holdout_f1),
    'cv_results': results_df.to_dict(orient='records'),
}

pred_df = pd.DataFrame({'true_label': y_test.values, 'predicted_label': pred})

with open_duckdb(output_db, read_only=False) as write_con:
    write_con.register('results_df', results_df)
    write_con.execute('CREATE OR REPLACE TABLE model_selection_results AS SELECT * FROM results_df')

    write_con.register('pred_df', pred_df)
    write_con.execute('CREATE OR REPLACE TABLE model_selection_predictions AS SELECT * FROM pred_df')

    write_con.execute('''
    INSERT OR REPLACE INTO ml_artifacts
    (artifact_id, notebook, artifact_type, model_name, metrics_json, artifact_blob)
    VALUES (?, ?, ?, ?, ?, ?)
    ''', [
        'model_selection_best_model',
        '06_model_selection',
        'model_selection_model',
        best_model_name,
        json.dumps(metrics),
        pickle.dumps(best_pipe),
    ])

results_df
            
